In [1]:
# ============================================================
# Fine-tuning済み Coarse-only 最終モデル：101ペア完全評価
#
# 出力：
# - pairwise_metrics_finetuned_coarse_only.csv
# - summary_metrics_finetuned_coarse_only.csv
#
# 評価：
# Dice, Jaccard, NCC, RMSE[HU], SSIM, MS-SSIM, FSIM
# 全画素Folding, valid-Body Folding, interior valid-Body Folding
# z=60–68 interior Folding, 肺野Dice, 肺野Folding
# ============================================================

import os
os.environ["VXM_BACKEND"] = "pytorch"

from pathlib import Path
import re
import gc
import math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import voxelmorph as vxm

from scipy.ndimage import (
    binary_fill_holes,
    binary_opening,
    binary_erosion,
    label,
)

from skimage.metrics import structural_similarity
from tqdm.auto import tqdm


backend:pytorch
Pytorch


In [2]:
# ------------------------------------------------------------
# 1. パス・設定
# ------------------------------------------------------------

TEST_DATA_ROOT = Path(
    r"D:\Saito\Data\TestData"
)

FINETUNED_MODEL_PATH = Path(
    r"D:\Saito\fullres_trilinear_ic_beta1e3_80k"
    r"\coarse_only_lll_ic_finetune_30k"
    r"\model_coarse_only_lll_ic_finetuned_30k_final.pth"
)

EVAL_DIR = (
    FINETUNED_MODEL_PATH.parent
    / "testdata_evaluation_finetuned_30k"
)

EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_NAME = "30k_coarse_only_finetuned"

FULL_SIZE = (128, 256, 256)
LOW_SIZE = (64, 128, 128)

HU_MIN = -1024.0
HU_MAX = 3071.0
BODY_THRESHOLD = 0.10

# 体表・画像端から除外する幅
INTERIOR_MARGIN_VOXELS = 5

# slice 64周辺の確認範囲
Z_CENTER = 64
Z_RADIUS = 4
Z_START = Z_CENTER - Z_RADIUS
Z_END = Z_CENTER + Z_RADIUS

DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

try:
    torch.set_float32_matmul_precision("high")
except AttributeError:
    pass

if not TEST_DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"TestDataが見つかりません:\n{TEST_DATA_ROOT}"
    )

if not FINETUNED_MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Fine-tuning済み重みが見つかりません:\n"
        f"{FINETUNED_MODEL_PATH}"
    )

print("Device      :", DEVICE)
print("TestData    :", TEST_DATA_ROOT)
print("Model       :", FINETUNED_MODEL_PATH)
print("Output dir  :", EVAL_DIR)



Device      : cuda:0
TestData    : D:\Saito\Data\TestData
Model       : D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\model_coarse_only_lll_ic_finetuned_30k_final.pth
Output dir  : D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\testdata_evaluation_finetuned_30k


In [3]:
# ------------------------------------------------------------
# 2. Haar DWT・モデル定義
# ------------------------------------------------------------

class Haar3DAnalysis(nn.Module):

    def __init__(self):
        super().__init__()

        h_low = torch.tensor(
            [1.0, 1.0],
            dtype=torch.float32,
        ) / math.sqrt(2.0)

        h_high = torch.tensor(
            [1.0, -1.0],
            dtype=torch.float32,
        ) / math.sqrt(2.0)

        filters = []

        for z_filter in [h_low, h_high]:
            for y_filter in [h_low, h_high]:
                for x_filter in [h_low, h_high]:
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)

        self.register_buffer(
            "weight",
            torch.stack(
                filters,
                dim=0,
            ).unsqueeze(1),
        )

    def forward(self, image):
        return F.conv3d(
            image,
            self.weight,
            stride=2,
            padding=0,
        )


def low_dvf_to_full_trilinear(flow_low):
    return F.interpolate(
        flow_low.float(),
        size=FULL_SIZE,
        mode="trilinear",
        align_corners=False,
    ) * 2.0


nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16],
]


def create_single_voxelmorph():
    return vxm.networks.VxmDense_128_256_256(
        FULL_SIZE,
        nb_features,
        nb_features,
        int_steps=0,
    )



In [4]:

# ------------------------------------------------------------
# 3. モデル読込
# ------------------------------------------------------------

try:
    checkpoint = torch.load(
        FINETUNED_MODEL_PATH,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    checkpoint = torch.load(
        FINETUNED_MODEL_PATH,
        map_location="cpu",
    )

state_dict = checkpoint.get(
    "model_state_dict",
    checkpoint,
)

state_dict = {
    key.replace("_orig_mod.", "", 1): value
    for key, value in state_dict.items()
}

analysis = Haar3DAnalysis().to(DEVICE)
analysis.eval()

model = create_single_voxelmorph().to(DEVICE)

model.load_state_dict(
    state_dict,
    strict=True,
)

model.eval()

transformer_full = vxm.layers.SpatialTransformer(
    FULL_SIZE
).to(DEVICE)

transformer_full.eval()

print("\nFine-tuned Coarse-only model loading completed.")


[64, 128, 128]

Fine-tuned Coarse-only model loading completed.


c:\Users\user\anaconda3\envs\nn\lib\site-packages\torch\functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [5]:
# ------------------------------------------------------------
# 4. データ読込
# ------------------------------------------------------------

def natural_sort_key(path):
    numbers = re.findall(
        r"\d+",
        path.name,
    )

    return [
        int(number)
        for number in numbers
    ] if numbers else [path.name]


def load_volume(path):
    path = Path(path)

    if path.suffix.lower() == ".npy":
        volume = np.load(path)

    elif path.suffix.lower() == ".npz":
        archive = np.load(path)

        preferred_keys = [
            "image",
            "volume",
            "ct",
            "data",
            "arr_0",
        ]

        selected_key = next(
            (
                key
                for key in preferred_keys
                if key in archive.files
            ),
            archive.files[0],
        )

        volume = archive[selected_key]

    else:
        raise ValueError(
            f"未対応形式です: {path}"
        )

    volume = np.asarray(volume).squeeze()

    if tuple(volume.shape) != FULL_SIZE:
        raise ValueError(
            f"shape={volume.shape}, "
            f"expected={FULL_SIZE}\n{path}"
        )

    return volume.astype(np.float32)


def find_fixed_moving_paths(pair_dir):
    files = sorted(
        list(pair_dir.rglob("*.npz"))
        + list(pair_dir.rglob("*.npy")),
        key=natural_sort_key,
    )

    fixed_candidates = [
        path
        for path in files
        if (
            "fixed" in path.name.lower()
            or "target" in path.name.lower()
        )
    ]

    moving_candidates = [
        path
        for path in files
        if (
            "moving" in path.name.lower()
            or "move" in path.name.lower()
            or "source" in path.name.lower()
        )
    ]

    if fixed_candidates and moving_candidates:
        return fixed_candidates[0], moving_candidates[0]

    if len(files) < 2:
        raise FileNotFoundError(
            f"Fixed / Moving画像を見つけられません:\n"
            f"{pair_dir}"
        )

    return files[0], files[1]


In [6]:
# ------------------------------------------------------------
# 5. 評価関数
# ------------------------------------------------------------

def compute_ncc(first, second):
    first = first.astype(np.float64)
    second = second.astype(np.float64)

    first = first - first.mean()
    second = second - second.mean()

    return float(
        np.sum(first * second)
        / (
            np.sqrt(
                np.sum(first ** 2)
                * np.sum(second ** 2)
            )
            + 1e-12
        )
    )


def make_body_mask(image):
    image_01 = np.clip(image, 0.0, 1.0)

    return binary_fill_holes(
        image_01 > BODY_THRESHOLD
    ).astype(bool)


def keep_largest_components(
    mask,
    n_components=2,
):
    labeled, count = label(mask)

    if count == 0:
        return np.zeros_like(
            mask,
            dtype=bool,
        )

    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0

    selected = np.argsort(sizes)[
        -n_components:
    ]

    return np.isin(
        labeled,
        selected,
    )


def make_lung_mask(image):
    image_01 = np.clip(image, 0.0, 1.0)

    image_hu = (
        image_01
        * (HU_MAX - HU_MIN)
        + HU_MIN
    )

    body = make_body_mask(image_01)

    lung_candidate = (
        (image_hu >= -1000.0)
        & (image_hu <= -300.0)
        & body
    )

    lung_candidate = binary_opening(
        lung_candidate,
        iterations=1,
    )

    return keep_largest_components(
        lung_candidate,
        n_components=2,
    ).astype(bool)


def dice_jaccard(mask_a, mask_b):
    intersection = np.logical_and(
        mask_a,
        mask_b,
    ).sum()

    union = np.logical_or(
        mask_a,
        mask_b,
    ).sum()

    dice = (
        2.0 * intersection
        / (mask_a.sum() + mask_b.sum() + 1e-12)
    )

    jaccard = (
        intersection
        / (union + 1e-12)
    )

    return float(dice), float(jaccard)


def compute_mean_ssim(fixed, moved):
    values = []

    for z in range(fixed.shape[0]):
        values.append(
            structural_similarity(
                np.clip(fixed[z], 0.0, 1.0),
                np.clip(moved[z], 0.0, 1.0),
                data_range=1.0,
            )
        )

    return float(np.mean(values))


def compute_ms_ssim_fsim(fixed, moved):
    try:
        import piq
    except ImportError:
        return np.nan, np.nan

    fixed_tensor = torch.from_numpy(
        np.clip(fixed, 0.0, 1.0).astype(np.float32)
    ).unsqueeze(1).to(DEVICE)

    moved_tensor = torch.from_numpy(
        np.clip(moved, 0.0, 1.0).astype(np.float32)
    ).unsqueeze(1).to(DEVICE)

    ms_ssim_value = np.nan
    fsim_value = np.nan

    try:
        ms_ssim_value = piq.multi_scale_ssim(
            moved_tensor,
            fixed_tensor,
            data_range=1.0,
            reduction="mean",
        ).item()
    except Exception as error:
        print("MS-SSIM skipped:", repr(error))

    try:
        middle = fixed.shape[0] // 2

        fsim_value = piq.fsim(
            moved_tensor[middle:middle + 1],
            fixed_tensor[middle:middle + 1],
            data_range=1.0,
            chromatic=False,
            reduction="mean",
        ).item()

    except TypeError:
        fsim_value = piq.fsim(
            moved_tensor[middle:middle + 1],
            fixed_tensor[middle:middle + 1],
            data_range=1.0,
            reduction="mean",
        ).item()

    except Exception as error:
        print("FSIM skipped:", repr(error))

    del fixed_tensor
    del moved_tensor

    return ms_ssim_value, fsim_value


def jacobian_determinant(flow_np):
    uz, uy, ux = flow_np

    duz_dz, duz_dy, duz_dx = np.gradient(
        uz,
        edge_order=1,
    )

    duy_dz, duy_dy, duy_dx = np.gradient(
        uy,
        edge_order=1,
    )

    dux_dz, dux_dy, dux_dx = np.gradient(
        ux,
        edge_order=1,
    )

    j11 = 1.0 + duz_dz
    j12 = duz_dy
    j13 = duz_dx

    j21 = duy_dz
    j22 = 1.0 + duy_dy
    j23 = duy_dx

    j31 = dux_dz
    j32 = dux_dy
    j33 = 1.0 + dux_dx

    return (
        j11 * (j22 * j33 - j23 * j32)
        - j12 * (j21 * j33 - j23 * j31)
        + j13 * (j21 * j32 - j22 * j31)
    ).astype(np.float32)


def valid_reference_mask(
    flow_full,
    margin=0,
):
    _, _, depth, height, width = flow_full.shape

    z = torch.arange(
        depth,
        device=flow_full.device,
        dtype=torch.float32,
    ).view(1, 1, depth, 1, 1)

    y = torch.arange(
        height,
        device=flow_full.device,
        dtype=torch.float32,
    ).view(1, 1, 1, height, 1)

    x = torch.arange(
        width,
        device=flow_full.device,
        dtype=torch.float32,
    ).view(1, 1, 1, 1, width)

    reference_z = z + flow_full[:, 0:1].float()
    reference_y = y + flow_full[:, 1:2].float()
    reference_x = x + flow_full[:, 2:3].float()

    margin = float(margin)

    valid = (
        (reference_z >= margin)
        & (reference_z <= float(depth - 1) - margin)
        & (reference_y >= margin)
        & (reference_y <= float(height - 1) - margin)
        & (reference_x >= margin)
        & (reference_x <= float(width - 1) - margin)
    )

    return valid[0, 0].cpu().numpy().astype(bool)


def warp_mask_nearest(mask_np, flow_full):
    mask_tensor = torch.from_numpy(
        mask_np.astype(np.float32)
    ).unsqueeze(0).unsqueeze(0).to(DEVICE)

    _, _, depth, height, width = flow_full.shape

    z = torch.arange(
        depth,
        device=DEVICE,
        dtype=torch.float32,
    )

    y = torch.arange(
        height,
        device=DEVICE,
        dtype=torch.float32,
    )

    x = torch.arange(
        width,
        device=DEVICE,
        dtype=torch.float32,
    )

    zz, yy, xx = torch.meshgrid(
        z, y, x,
        indexing="ij",
    )

    grid = torch.stack(
        [zz, yy, xx],
        dim=0,
    ).unsqueeze(0)

    grid = grid + flow_full.float()

    grid[:, 0] = (
        2.0 * grid[:, 0]
        / max(depth - 1, 1)
        - 1.0
    )

    grid[:, 1] = (
        2.0 * grid[:, 1]
        / max(height - 1, 1)
        - 1.0
    )

    grid[:, 2] = (
        2.0 * grid[:, 2]
        / max(width - 1, 1)
        - 1.0
    )

    grid = grid.permute(
        0, 2, 3, 4, 1
    )[..., [2, 1, 0]]

    warped = F.grid_sample(
        mask_tensor,
        grid,
        mode="nearest",
        padding_mode="zeros",
        align_corners=True,
    )

    return (
        warped[0, 0].cpu().numpy() > 0.5
    )


def folding_percent(jacobian, mask):
    if mask.sum() == 0:
        return np.nan

    return float(
        np.mean(jacobian[mask] <= 0.0)
        * 100.0
    )



In [7]:

# ------------------------------------------------------------
# 6. 1ペア評価
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_pair(pair_dir):
    fixed_path, moving_path = find_fixed_moving_paths(
        pair_dir
    )

    fixed_np = load_volume(fixed_path)
    moving_np = load_volume(moving_path)

    fixed_tensor = torch.from_numpy(
        fixed_np
    ).unsqueeze(0).unsqueeze(0).to(
        DEVICE,
        dtype=torch.float32,
    )

    moving_tensor = torch.from_numpy(
        moving_np
    ).unsqueeze(0).unsqueeze(0).to(
        DEVICE,
        dtype=torch.float32,
    )

    moving_bands = analysis(moving_tensor)
    fixed_bands = analysis(fixed_tensor)

    flow_low = model(
        moving_bands,
        fixed_bands,
    )

    flow_full = low_dvf_to_full_trilinear(
        flow_low
    )

    moved_tensor = transformer_full(
        moving_tensor,
        flow_full,
    )

    moved_np = moved_tensor[0, 0].cpu().numpy()
    flow_np = flow_full[0].cpu().numpy()

    fixed_body = make_body_mask(fixed_np)
    moving_body = make_body_mask(moving_np)
    moved_body = make_body_mask(moved_np)

    fixed_lung = make_lung_mask(fixed_np)
    moving_lung = make_lung_mask(moving_np)

    moved_lung = warp_mask_nearest(
        moving_lung,
        flow_full,
    )

    jacobian = jacobian_determinant(flow_np)

    valid_body = (
        fixed_body
        & valid_reference_mask(
            flow_full,
            margin=0,
        )
    )

    interior_body = binary_erosion(
        fixed_body,
        iterations=INTERIOR_MARGIN_VOXELS,
    ).astype(bool)

    interior_valid_body = (
        interior_body
        & valid_reference_mask(
            flow_full,
            margin=INTERIOR_MARGIN_VOXELS,
        )
    )

    z_window = np.zeros_like(
        fixed_body,
        dtype=bool,
    )

    z0 = max(0, Z_START)
    z1 = min(FULL_SIZE[0] - 1, Z_END)

    z_window[z0:z1 + 1] = True

    interior_valid_body_z64 = (
        interior_valid_body
        & z_window
    )

    mse_before = np.mean(
        (moving_np - fixed_np) ** 2
    )

    mse_after = np.mean(
        (moved_np - fixed_np) ** 2
    )

    ms_ssim_value, fsim_value = compute_ms_ssim_fsim(
        fixed_np,
        moved_np,
    )

    dice_before, jaccard_before = dice_jaccard(
        fixed_body,
        moving_body,
    )

    dice_after, jaccard_after = dice_jaccard(
        fixed_body,
        moved_body,
    )

    result = {
        "model": MODEL_NAME,
        "pair": pair_dir.name,
        "fixed_file": fixed_path.name,
        "moving_file": moving_path.name,

        "dice_before": dice_before,
        "dice_after": dice_after,
        "jaccard_before": jaccard_before,
        "jaccard_after": jaccard_after,

        "ncc_before": compute_ncc(
            fixed_np,
            moving_np,
        ),
        "ncc_after": compute_ncc(
            fixed_np,
            moved_np,
        ),

        "rmse_hu_before": float(
            np.sqrt(mse_before)
            * (HU_MAX - HU_MIN)
        ),
        "rmse_hu_after": float(
            np.sqrt(mse_after)
            * (HU_MAX - HU_MIN)
        ),

        "ssim_after": compute_mean_ssim(
            fixed_np,
            moved_np,
        ),

        "ms_ssim_after": ms_ssim_value,
        "fsim_after": fsim_value,

        # 従来どおり：全画素
        "raw_full_folding_percent": float(
            np.mean(jacobian <= 0.0) * 100.0
        ),

        # Body内かつMoving参照座標が画像内
        "valid_body_folding_percent": folding_percent(
            jacobian,
            valid_body,
        ),

        # 体表・画像端から5 voxelを除外
        "interior_valid_body_folding_percent": (
            folding_percent(
                jacobian,
                interior_valid_body,
            )
        ),

        # z=60–68に限定
        "interior_valid_body_folding_z60_to_z68_percent": (
            folding_percent(
                jacobian,
                interior_valid_body_z64,
            )
        ),

        "jacobian_min": float(jacobian.min()),

        "lung_dice_after": dice_jaccard(
            fixed_lung,
            moved_lung,
        )[0],

        "lung_folding_percent": folding_percent(
            jacobian,
            fixed_lung,
        ),

        "valid_body_fraction_percent": float(
            100.0
            * valid_body.sum()
            / max(fixed_body.sum(), 1)
        ),

        "interior_valid_body_fraction_percent": float(
            100.0
            * interior_valid_body.sum()
            / max(fixed_body.sum(), 1)
        ),

        "flow_magnitude_mean_voxel": float(
            np.sqrt(
                np.sum(flow_np ** 2, axis=0)
            ).mean()
        ),
    }

    del fixed_tensor
    del moving_tensor
    del moving_bands
    del fixed_bands
    del flow_low
    del flow_full
    del moved_tensor

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result



In [8]:
# ------------------------------------------------------------
# 7. 全101ペア評価
# ------------------------------------------------------------

pair_dirs = sorted(
    [
        path
        for path in TEST_DATA_ROOT.iterdir()
        if path.is_dir()
        and path.name.lower().startswith("pair")
    ],
    key=natural_sort_key,
)

if len(pair_dirs) == 0:
    raise FileNotFoundError(
        f"pairフォルダがありません:\n{TEST_DATA_ROOT}"
    )

print(f"\n評価pair数: {len(pair_dirs)}")

rows = []

for index, pair_dir in enumerate(
    tqdm(pair_dirs, desc="Fine-tuned Coarse-only evaluation"),
    start=1,
):
    try:
        result = evaluate_pair(pair_dir)
        rows.append(result)

        print(
            f"[{index:03d}/{len(pair_dirs):03d}] "
            f"{pair_dir.name} | "
            f"Dice={result['dice_after']:.5f} | "
            f"NCC={result['ncc_after']:.5f} | "
            f"rawFold={result['raw_full_folding_percent']:.4f}% | "
            f"interiorFold={result['interior_valid_body_folding_percent']:.4f}%"
        )

    except Exception as error:
        print(
            f"[Skip] {pair_dir.name}: {repr(error)}"
        )



評価pair数: 101


Fine-tuned Coarse-only evaluation:   0%|          | 0/101 [00:00<?, ?it/s]

[001/101] pair1 | Dice=0.99536 | NCC=0.99326 | rawFold=0.7392% | interiorFold=0.2593%
[002/101] pair2 | Dice=0.99412 | NCC=0.99223 | rawFold=0.4451% | interiorFold=0.7323%
[003/101] pair3 | Dice=0.99406 | NCC=0.99182 | rawFold=0.3438% | interiorFold=0.2528%
[004/101] pair4 | Dice=0.99477 | NCC=0.99322 | rawFold=0.0746% | interiorFold=0.1718%
[005/101] pair5 | Dice=0.99595 | NCC=0.99295 | rawFold=0.1501% | interiorFold=0.3148%
[006/101] pair6 | Dice=0.99624 | NCC=0.99375 | rawFold=0.1746% | interiorFold=0.2364%
[007/101] pair7 | Dice=0.99595 | NCC=0.99323 | rawFold=0.7353% | interiorFold=0.1744%
[008/101] pair8 | Dice=0.99589 | NCC=0.99359 | rawFold=0.7976% | interiorFold=0.2013%
[009/101] pair9 | Dice=0.99509 | NCC=0.99286 | rawFold=0.2607% | interiorFold=0.3780%
[010/101] pair10 | Dice=0.99438 | NCC=0.99227 | rawFold=0.3835% | interiorFold=0.7612%
[011/101] pair11 | Dice=0.99522 | NCC=0.99208 | rawFold=0.4330% | interiorFold=0.1095%
[012/101] pair12 | Dice=0.99373 | NCC=0.99194 | rawF

In [9]:

# ------------------------------------------------------------
# 8. Pair-wise CSV
# ------------------------------------------------------------

pairwise_df = pd.DataFrame(rows)

pairwise_csv = (
    EVAL_DIR
    / "pairwise_metrics_finetuned_coarse_only.csv"
)

pairwise_df.to_csv(
    pairwise_csv,
    index=False,
    encoding="utf-8-sig",
)

display(pairwise_df.round(6))

print("\nPair-wise CSV:")
print(pairwise_csv)


,model,pair,fixed_file,moving_file,dice_before,dice_after,jaccard_before,jaccard_after,ncc_before,ncc_after,...,raw_full_folding_percent,valid_body_folding_percent,interior_valid_body_folding_percent,interior_valid_body_folding_z60_to_z68_percent,jacobian_min,lung_dice_after,lung_folding_percent,valid_body_fraction_percent,interior_valid_body_fraction_percent,flow_magnitude_mean_voxel
0,30k_coarse_only_finetuned,pair1,registered_masked_A_MIC00197_2018_0428.npz,fixed_masked_B_MIC00197_2019_0406.npz,0.961797,0.995362,0.926405,0.990766,0.928469,0.993257,...,0.739229,0.300952,0.259330,0.008706,-3.709836,0.893092,0.118460,99.571015,83.405730,1.348287
1,30k_coarse_only_finetuned,pair2,registered_masked_A_MIC00761_2019_0727.npz,fixed_masked_B_MIC00761_2020_0926.npz,0.981110,0.994116,0.962920,0.988301,0.962952,0.992235,...,0.445092,0.848955,0.732283,0.069298,-5.062313,0.896702,0.002335,99.745335,82.743739,1.196022
2,30k_coarse_only_finetuned,pair3,registered_masked_A_MIC03568_2018_0825.npz,fixed_masked_B_MIC03568_2019_0824.npz,0.963118,0.994057,0.928860,0.988184,0.928717,0.991821,...,0.343764,0.243183,0.252827,0.132985,-3.023296,0.842017,0.047285,99.344100,82.652642,1.317584
3,30k_coarse_only_finetuned,pair4,registered_masked_A_MIC03568_2018_0825.npz,fixed_masked_B_MIC03568_2020_0829.npz,0.982659,0.994768,0.965908,0.989591,0.957978,0.993216,...,0.074601,0.166762,0.171781,0.028936,-1.250471,0.843378,0.008412,99.813316,82.595646,1.205959
4,30k_coarse_only_finetuned,pair5,registered_masked_A_MIC03568_2019_0824.npz,fixed_masked_B_MIC03568_2020_0829.npz,0.986525,0.995950,0.973408,0.991932,0.959973,0.992949,...,0.150120,0.325566,0.314843,0.151760,-1.851350,0.823556,0.025732,99.900783,82.415131,1.382915
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,30k_coarse_only_finetuned,pair97,registered_masked_A_MIC45438_2019_0903.npz,fixed_masked_B_MIC45438_2020_0824.npz,0.979977,0.995550,0.960741,0.991139,0.947270,0.993477,...,0.269413,0.547465,0.570696,0.032274,-1.863317,0.772483,0.703898,99.826144,82.742414,1.739092
97,30k_coarse_only_finetuned,pair98,registered_masked_A_MIC52743_2018_1213.npz,fixed_masked_B_MIC52743_2020_0125.npz,0.978022,0.994715,0.956990,0.989485,0.952752,0.994009,...,0.134385,0.250556,0.183615,0.066265,-2.327193,0.865215,0.032687,99.850940,83.118550,1.474429
98,30k_coarse_only_finetuned,pair99,registered_masked_A_MIC53652_2019_1213.npz,fixed_masked_B_MIC53652_2020_1219.npz,0.981859,0.995778,0.964364,0.991592,0.947066,0.991595,...,0.223482,0.399139,0.360772,0.228126,-3.746541,0.836289,0.086755,99.655327,83.620737,1.739125
99,30k_coarse_only_finetuned,pair100,registered_masked_A_MIC53770_2018_1128.npz,fixed_masked_B_MIC53770_2019_1111.npz,0.980435,0.995213,0.961621,0.990471,0.954214,0.992648,...,0.147796,0.323363,0.289747,0.003718,-3.644271,0.928089,0.102263,99.434115,83.265800,1.289984



Pair-wise CSV:
D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\testdata_evaluation_finetuned_30k\pairwise_metrics_finetuned_coarse_only.csv


In [10]:
# ------------------------------------------------------------
# 9. 平均表
# ------------------------------------------------------------

if len(pairwise_df) == 0:
    raise RuntimeError(
        "評価成功pairが0件です。"
    )

raw_worst = pairwise_df.loc[
    pairwise_df[
        "raw_full_folding_percent"
    ].idxmax()
]

valid_worst = pairwise_df.loc[
    pairwise_df[
        "valid_body_folding_percent"
    ].idxmax()
]

interior_worst = pairwise_df.loc[
    pairwise_df[
        "interior_valid_body_folding_percent"
    ].idxmax()
]

z64_worst = pairwise_df.loc[
    pairwise_df[
        "interior_valid_body_folding_z60_to_z68_percent"
    ].idxmax()
]

summary_df = pd.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "n_pairs": int(len(pairwise_df)),

            "dice_before_mean": pairwise_df[
                "dice_before"
            ].mean(),
            "dice_after_mean": pairwise_df[
                "dice_after"
            ].mean(),
            "jaccard_after_mean": pairwise_df[
                "jaccard_after"
            ].mean(),

            "ncc_before_mean": pairwise_df[
                "ncc_before"
            ].mean(),
            "ncc_after_mean": pairwise_df[
                "ncc_after"
            ].mean(),

            "rmse_hu_before_mean": pairwise_df[
                "rmse_hu_before"
            ].mean(),
            "rmse_hu_after_mean": pairwise_df[
                "rmse_hu_after"
            ].mean(),

            "ssim_after_mean": pairwise_df[
                "ssim_after"
            ].mean(),
            "ms_ssim_after_mean": pairwise_df[
                "ms_ssim_after"
            ].mean(),
            "fsim_after_mean": pairwise_df[
                "fsim_after"
            ].mean(),

            "raw_full_folding_mean_percent": pairwise_df[
                "raw_full_folding_percent"
            ].mean(),
            "raw_full_folding_max_percent": pairwise_df[
                "raw_full_folding_percent"
            ].max(),
            "raw_full_worst_pair": raw_worst["pair"],

            "valid_body_folding_mean_percent": pairwise_df[
                "valid_body_folding_percent"
            ].mean(),
            "valid_body_folding_max_percent": pairwise_df[
                "valid_body_folding_percent"
            ].max(),
            "valid_body_worst_pair": valid_worst["pair"],

            "interior_valid_body_folding_mean_percent": pairwise_df[
                "interior_valid_body_folding_percent"
            ].mean(),
            "interior_valid_body_folding_max_percent": pairwise_df[
                "interior_valid_body_folding_percent"
            ].max(),
            "interior_valid_body_worst_pair": interior_worst["pair"],

            "interior_valid_body_folding_z60_to_z68_mean_percent": (
                pairwise_df[
                    "interior_valid_body_folding_z60_to_z68_percent"
                ].mean()
            ),
            "interior_valid_body_folding_z60_to_z68_max_percent": (
                pairwise_df[
                    "interior_valid_body_folding_z60_to_z68_percent"
                ].max()
            ),
            "interior_valid_body_folding_z60_to_z68_worst_pair": (
                z64_worst["pair"]
            ),

            "lung_dice_after_mean": pairwise_df[
                "lung_dice_after"
            ].mean(),
            "lung_folding_mean_percent": pairwise_df[
                "lung_folding_percent"
            ].mean(),

            "flow_magnitude_mean_voxel": pairwise_df[
                "flow_magnitude_mean_voxel"
            ].mean(),
        }
    ]
)

summary_csv = (
    EVAL_DIR
    / "summary_metrics_finetuned_coarse_only.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False,
    encoding="utf-8-sig",
)

print("\n================ FINAL SUMMARY ================\n")
display(summary_df.round(6))

print("\nSummary CSV:")
print(summary_csv)

print("\nCompleted.")


================ FINAL SUMMARY ================



,model,n_pairs,dice_before_mean,dice_after_mean,jaccard_after_mean,ncc_before_mean,ncc_after_mean,rmse_hu_before_mean,rmse_hu_after_mean,ssim_after_mean,...,valid_body_worst_pair,interior_valid_body_folding_mean_percent,interior_valid_body_folding_max_percent,interior_valid_body_worst_pair,interior_valid_body_folding_z60_to_z68_mean_percent,interior_valid_body_folding_z60_to_z68_max_percent,interior_valid_body_folding_z60_to_z68_worst_pair,lung_dice_after_mean,lung_folding_mean_percent,flow_magnitude_mean_voxel
0,30k_coarse_only_finetuned,101,0.959418,0.991369,0.983194,0.920009,0.987036,411.771446,154.169764,0.910547,...,pair79,0.644943,2.393665,pair78,0.282361,3.237829,pair90,0.790941,0.531124,1.746876



Summary CSV:
D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\testdata_evaluation_finetuned_30k\summary_metrics_finetuned_coarse_only.csv

Completed.


In [11]:
display(summary_df.T)

,0
model,30k_coarse_only_finetuned
n_pairs,101
dice_before_mean,0.959418
dice_after_mean,0.991369
jaccard_after_mean,0.983194
ncc_before_mean,0.920009
ncc_after_mean,0.987036
rmse_hu_before_mean,411.771446
rmse_hu_after_mean,154.169764
ssim_after_mean,0.910547


In [12]:
# ============================================================
# Fine-tuning後Coarse-only：Moving / Fixed / Moved画像を保存
# ============================================================

import gc
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

# None: 全101ペアを保存
# 例 ["pair61", "pair78"] のようにすると指定pairだけ保存
PAIR_NAMES_TO_SAVE = None

# axial slice
SLICE_INDEX = 64

# 3DのMoved画像も保存するか
# Trueにするとファイル容量が大きくなるため、通常はFalse推奨
SAVE_MOVED_VOLUME_NPZ = False

required_names = [
    "TEST_DATA_ROOT",
    "EVAL_DIR",
    "DEVICE",
    "analysis",
    "model",
    "transformer_full",
    "low_dvf_to_full_trilinear",
    "load_volume",
    "find_fixed_moving_paths",
]

missing = [
    name
    for name in required_names
    if name not in globals()
]

if missing:
    raise NameError(
        "先にFine-tuning後モデルの完全評価セルを実行してください。\n"
        + ", ".join(missing)
    )

IMAGE_OUTPUT_DIR = EVAL_DIR / "axial_slice64_images"

IMAGE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if PAIR_NAMES_TO_SAVE is None:
    pair_dirs = sorted(
        [
            path
            for path in TEST_DATA_ROOT.iterdir()
            if path.is_dir()
            and path.name.lower().startswith("pair")
        ],
        key=natural_sort_key,
    )
else:
    pair_dirs = [
        TEST_DATA_ROOT / pair_name
        for pair_name in PAIR_NAMES_TO_SAVE
    ]

print("保存pair数 :", len(pair_dirs))
print("保存先     :", IMAGE_OUTPUT_DIR)


@torch.no_grad()
def infer_moved_image(
    fixed_np,
    moving_np,
):
    fixed_tensor = torch.from_numpy(
        fixed_np.astype(np.float32)
    ).unsqueeze(0).unsqueeze(0).to(DEVICE)

    moving_tensor = torch.from_numpy(
        moving_np.astype(np.float32)
    ).unsqueeze(0).unsqueeze(0).to(DEVICE)

    moving_bands = analysis(moving_tensor)
    fixed_bands = analysis(fixed_tensor)

    flow_low = model(
        moving_bands,
        fixed_bands,
    )

    flow_full = low_dvf_to_full_trilinear(
        flow_low
    )

    moved_tensor = transformer_full(
        moving_tensor,
        flow_full,
    )

    moved_np = moved_tensor[
        0,
        0,
    ].cpu().numpy()

    del fixed_tensor
    del moving_tensor
    del moving_bands
    del fixed_bands
    del flow_low
    del flow_full
    del moved_tensor

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return moved_np


for index, pair_dir in enumerate(
    pair_dirs,
    start=1,
):
    if not pair_dir.is_dir():
        print(f"[Skip] pairフォルダなし: {pair_dir}")
        continue

    try:
        fixed_path, moving_path = find_fixed_moving_paths(
            pair_dir
        )

        fixed_np = load_volume(fixed_path)
        moving_np = load_volume(moving_path)

        if not (
            0 <= SLICE_INDEX < fixed_np.shape[0]
        ):
            raise ValueError(
                f"slice={SLICE_INDEX} が範囲外です"
            )

        moved_np = infer_moved_image(
            fixed_np=fixed_np,
            moving_np=moving_np,
        )

        moving_slice = moving_np[SLICE_INDEX]
        fixed_slice = fixed_np[SLICE_INDEX]
        moved_slice = moved_np[SLICE_INDEX]

        difference = np.abs(
            fixed_slice - moved_slice
        )

        vmin, vmax = np.percentile(
            np.concatenate(
                [
                    moving_slice.ravel(),
                    fixed_slice.ravel(),
                    moved_slice.ravel(),
                ]
            ),
            [1, 99],
        )

        diff_max = np.percentile(
            difference,
            99,
        )

        fig, axes = plt.subplots(
            1,
            4,
            figsize=(18, 5),
            constrained_layout=True,
        )

        images = [
            moving_slice,
            fixed_slice,
            moved_slice,
            difference,
        ]

        titles = [
            "Moving",
            "Fixed",
            "Fine-tuned Coarse-only Moved",
            "| Fixed − Moved |",
        ]

        for column, (
            image,
            title,
        ) in enumerate(
            zip(images, titles)
        ):
            if column < 3:
                axes[column].imshow(
                    image,
                    cmap="gray",
                    vmin=vmin,
                    vmax=vmax,
                )
            else:
                axes[column].imshow(
                    image,
                    cmap="magma",
                    vmin=0.0,
                    vmax=diff_max,
                )

            axes[column].set_title(
                title,
                fontsize=12,
            )

            axes[column].axis("off")

        fig.suptitle(
            f"{pair_dir.name} | axial slice {SLICE_INDEX} | "
            "Fine-tuned Coarse-only",
            fontsize=15,
        )

        png_path = (
            IMAGE_OUTPUT_DIR
            / (
                f"{pair_dir.name}_slice"
                f"{SLICE_INDEX:03d}_fine_tuned_coarse_only.png"
            )
        )

        fig.savefig(
            png_path,
            dpi=180,
            bbox_inches="tight",
        )

        plt.close(fig)

        if SAVE_MOVED_VOLUME_NPZ:
            npz_path = (
                IMAGE_OUTPUT_DIR
                / (
                    f"{pair_dir.name}_"
                    "fine_tuned_coarse_only_moved.npz"
                )
            )

            np.savez_compressed(
                npz_path,
                moved=moved_np.astype(np.float32),
            )

        print(
            f"[{index:03d}/{len(pair_dirs):03d}] "
            f"saved: {png_path.name}"
        )

    except Exception as error:
        print(
            f"[Skip] {pair_dir.name}: {repr(error)}"
        )

print("\n画像保存完了:")
print(IMAGE_OUTPUT_DIR)

保存pair数 : 101
保存先     : D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\testdata_evaluation_finetuned_30k\axial_slice64_images
[001/101] saved: pair1_slice064_fine_tuned_coarse_only.png
[002/101] saved: pair2_slice064_fine_tuned_coarse_only.png
[003/101] saved: pair3_slice064_fine_tuned_coarse_only.png
[004/101] saved: pair4_slice064_fine_tuned_coarse_only.png
[005/101] saved: pair5_slice064_fine_tuned_coarse_only.png
[006/101] saved: pair6_slice064_fine_tuned_coarse_only.png
[007/101] saved: pair7_slice064_fine_tuned_coarse_only.png
[008/101] saved: pair8_slice064_fine_tuned_coarse_only.png
[009/101] saved: pair9_slice064_fine_tuned_coarse_only.png
[010/101] saved: pair10_slice064_fine_tuned_coarse_only.png
[011/101] saved: pair11_slice064_fine_tuned_coarse_only.png
[012/101] saved: pair12_slice064_fine_tuned_coarse_only.png
[013/101] saved: pair13_slice064_fine_tuned_coarse_only.png
[014/101] saved: pair14_slice064_fine_tuned_coarse_only.png
[015/101] saved:

In [13]:
# ============================================================
# Coarse-only LLL + IC fine-tuned model:
# TestData 101ペア 完全評価
#
# 評価:
# Dice / Jaccard / NCC / RMSE[HU] / SSIM / MS-SSIM / FSIM
# raw / valid-body / interior valid-body / z60-68 Folding
# Jacobian最小値 / lung Dice / lung Folding / 平均DVF大きさ
# IC MSE（学習時と同じ low-DVF 空間での逆写像整合性）
#
# 出力:
# - pairwise_metrics_coarse_only_lll_ic_finetuned.csv
# - summary_metrics_coarse_only_lll_ic_finetuned.csv
# - pair_images/*.png
# ============================================================

import os
os.environ["VXM_BACKEND"] = "pytorch"

from pathlib import Path
import re
import gc
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import voxelmorph as vxm

from scipy.ndimage import (
    binary_fill_holes,
    binary_opening,
    binary_erosion,
    label,
)
from skimage.metrics import structural_similarity
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 1. パス・基本設定
# ------------------------------------------------------------

TEST_DATA_ROOT = Path(
    r"D:\Saito\Data\TestData"
)

MODEL_PATH = Path(
    r"D:\Saito\fullres_trilinear_ic_beta1e3_80k"
    r"\coarse_only_lll_ic_finetune_30k"
    r"\model_coarse_only_lll_ic_finetuned_30k_final.pth"
)

EVAL_DIR = (
    MODEL_PATH.parent
    / "testdata_evaluation_coarse_only_lll_ic_finetuned_30k"
)

IMAGE_DIR = EVAL_DIR / "pair_images"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "30k_coarse_only_lll_ic_finetuned"

FULL_SIZE = (128, 256, 256)
LOW_SIZE = (64, 128, 128)

HU_MIN = -1024.0
HU_MAX = 3071.0

BODY_THRESHOLD = 0.10
INTERIOR_MARGIN_VOXELS = 5

Z_CENTER = 64
Z_RADIUS = 4
Z_START = Z_CENTER - Z_RADIUS
Z_END = Z_CENTER + Z_RADIUS

# True: 全101ペアの4-panel画像を保存
SAVE_ALL_PAIR_IMAGES = True

DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

try:
    torch.set_float32_matmul_precision("high")
except AttributeError:
    pass

if not TEST_DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"TestDataが見つかりません:\n{TEST_DATA_ROOT}"
    )

if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Fine-tuning済み重みが見つかりません:\n{MODEL_PATH}"
    )

print("Device     :", DEVICE)
print("TestData   :", TEST_DATA_ROOT)
print("Model path :", MODEL_PATH)
print("Output dir :", EVAL_DIR)
print("Direction  : Moving -> Fixed")


# ------------------------------------------------------------
# 2. Haar解析・DVF変換・モデル
# ------------------------------------------------------------

class Haar3DAnalysis(nn.Module):

    def __init__(self):
        super().__init__()

        h_low = torch.tensor(
            [1.0, 1.0],
            dtype=torch.float32,
        ) / math.sqrt(2.0)

        h_high = torch.tensor(
            [1.0, -1.0],
            dtype=torch.float32,
        ) / math.sqrt(2.0)

        filters = []

        for z_filter in [h_low, h_high]:
            for y_filter in [h_low, h_high]:
                for x_filter in [h_low, h_high]:
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)

        self.register_buffer(
            "weight",
            torch.stack(filters, dim=0).unsqueeze(1),
        )

    def forward(self, image):
        return F.conv3d(
            image,
            self.weight,
            stride=2,
            padding=0,
        )


def low_dvf_to_full_trilinear(flow_low):
    return F.interpolate(
        flow_low.float(),
        size=FULL_SIZE,
        mode="trilinear",
        align_corners=False,
    ) * 2.0


nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16],
]


def create_single_voxelmorph():
    return vxm.networks.VxmDense_128_256_256(
        FULL_SIZE,
        nb_features,
        nb_features,
        int_steps=0,
    )


# ------------------------------------------------------------
# 3. 重み読込
# ------------------------------------------------------------

try:
    checkpoint = torch.load(
        MODEL_PATH,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    checkpoint = torch.load(
        MODEL_PATH,
        map_location="cpu",
    )

state_dict = checkpoint.get(
    "model_state_dict",
    checkpoint,
)

state_dict = {
    key.replace("_orig_mod.", "", 1): value
    for key, value in state_dict.items()
}

analysis = Haar3DAnalysis().to(DEVICE).eval()

model = create_single_voxelmorph().to(DEVICE)
model.load_state_dict(state_dict, strict=True)
model.eval()

transformer_full = vxm.layers.SpatialTransformer(
    FULL_SIZE
).to(DEVICE).eval()

# IC MSE計算用：学習時と同じlow-DVF空間のTransformer
transformer_low = vxm.layers.SpatialTransformer(
    LOW_SIZE
).to(DEVICE).eval()

print("\nModel loading completed.")
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print("Architecture    :", checkpoint.get("architecture", "unknown"))
print("Loss            :", checkpoint.get("loss_name", "unknown"))


# ------------------------------------------------------------
# 4. データ読込・ファイル探索
# ------------------------------------------------------------

def natural_sort_key(path):
    parts = re.findall(r"\d+", Path(path).name)
    return [int(value) for value in parts] if parts else [Path(path).name]


def load_volume(path):
    path = Path(path)

    if path.suffix.lower() == ".npy":
        volume = np.load(path)

    elif path.suffix.lower() == ".npz":
        archive = np.load(path)

        preferred_keys = [
            "image",
            "volume",
            "arr_0",
            "data",
        ]

        key = next(
            (
                candidate
                for candidate in preferred_keys
                if candidate in archive.files
            ),
            archive.files[0],
        )

        volume = archive[key]

    else:
        raise ValueError(f"未対応形式です: {path}")

    volume = np.asarray(volume).squeeze()

    if tuple(volume.shape) != FULL_SIZE:
        raise ValueError(
            f"shape={volume.shape}, expected={FULL_SIZE}\n{path}"
        )

    return volume.astype(np.float32)


def find_fixed_moving_paths(pair_dir):
    files = sorted(
        list(pair_dir.rglob("*.npz"))
        + list(pair_dir.rglob("*.npy")),
        key=natural_sort_key,
    )

    fixed_candidates = [
        path
        for path in files
        if (
            "fixed" in path.name.lower()
            or "target" in path.name.lower()
        )
    ]

    moving_candidates = [
        path
        for path in files
        if (
            "moving" in path.name.lower()
            or "move" in path.name.lower()
            or "source" in path.name.lower()
            or "registered" in path.name.lower()
        )
    ]

    if fixed_candidates and moving_candidates:
        return fixed_candidates[0], moving_candidates[0]

    raise FileNotFoundError(
        "Fixed / Moving画像を名前から特定できません。\n"
        f"{pair_dir}\n"
        f"検出ファイル: {[path.name for path in files]}"
    )


# ------------------------------------------------------------
# 5. 指標関数
# ------------------------------------------------------------

def compute_ncc(first, second):
    first = first.astype(np.float64)
    second = second.astype(np.float64)

    first = first - first.mean()
    second = second - second.mean()

    return float(
        np.sum(first * second)
        / (
            np.sqrt(
                np.sum(first ** 2)
                * np.sum(second ** 2)
            )
            + 1e-12
        )
    )


def make_body_mask(image):
    return binary_fill_holes(
        np.clip(image, 0.0, 1.0) > BODY_THRESHOLD
    ).astype(bool)


def keep_largest_components(mask, n_components=2):
    labeled, count = label(mask)

    if count == 0:
        return np.zeros_like(mask, dtype=bool)

    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0

    selected = np.argsort(sizes)[-n_components:]

    return np.isin(labeled, selected)


def make_lung_mask(image):
    image_hu = (
        np.clip(image, 0.0, 1.0)
        * (HU_MAX - HU_MIN)
        + HU_MIN
    )

    body = make_body_mask(image)

    lung_candidate = (
        (image_hu >= -1000.0)
        & (image_hu <= -300.0)
        & body
    )

    lung_candidate = binary_opening(
        lung_candidate,
        iterations=1,
    )

    return keep_largest_components(
        lung_candidate,
        n_components=2,
    ).astype(bool)


def dice_jaccard(mask_a, mask_b):
    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()

    dice = (
        2.0 * intersection
        / (mask_a.sum() + mask_b.sum() + 1e-12)
    )

    jaccard = intersection / (union + 1e-12)

    return float(dice), float(jaccard)


def compute_mean_ssim(fixed, moved):
    values = []

    for z in range(fixed.shape[0]):
        values.append(
            structural_similarity(
                np.clip(fixed[z], 0.0, 1.0),
                np.clip(moved[z], 0.0, 1.0),
                data_range=1.0,
            )
        )

    return float(np.mean(values))


def compute_ms_ssim_fsim(fixed, moved):
    try:
        import piq
    except ImportError:
        return np.nan, np.nan

    fixed_tensor = torch.from_numpy(
        np.clip(fixed, 0.0, 1.0).astype(np.float32)
    ).unsqueeze(1).to(DEVICE)

    moved_tensor = torch.from_numpy(
        np.clip(moved, 0.0, 1.0).astype(np.float32)
    ).unsqueeze(1).to(DEVICE)

    ms_ssim_value = np.nan
    fsim_value = np.nan

    try:
        ms_ssim_value = piq.multi_scale_ssim(
            moved_tensor,
            fixed_tensor,
            data_range=1.0,
            reduction="mean",
        ).item()
    except Exception as error:
        print("MS-SSIM skipped:", repr(error))

    try:
        middle = fixed.shape[0] // 2

        fsim_value = piq.fsim(
            moved_tensor[middle:middle + 1],
            fixed_tensor[middle:middle + 1],
            data_range=1.0,
            chromatic=False,
            reduction="mean",
        ).item()

    except TypeError:
        fsim_value = piq.fsim(
            moved_tensor[middle:middle + 1],
            fixed_tensor[middle:middle + 1],
            data_range=1.0,
            reduction="mean",
        ).item()

    except Exception as error:
        print("FSIM skipped:", repr(error))

    del fixed_tensor
    del moved_tensor

    return ms_ssim_value, fsim_value


def jacobian_determinant(flow_np):
    uz, uy, ux = flow_np

    duz_dz, duz_dy, duz_dx = np.gradient(
        uz,
        edge_order=1,
    )

    duy_dz, duy_dy, duy_dx = np.gradient(
        uy,
        edge_order=1,
    )

    dux_dz, dux_dy, dux_dx = np.gradient(
        ux,
        edge_order=1,
    )

    j11 = 1.0 + duz_dz
    j12 = duz_dy
    j13 = duz_dx

    j21 = duy_dz
    j22 = 1.0 + duy_dy
    j23 = duy_dx

    j31 = dux_dz
    j32 = dux_dy
    j33 = 1.0 + dux_dx

    return (
        j11 * (j22 * j33 - j23 * j32)
        - j12 * (j21 * j33 - j23 * j31)
        + j13 * (j21 * j32 - j22 * j31)
    ).astype(np.float32)


def valid_reference_mask(flow_full, margin=0):
    _, _, depth, height, width = flow_full.shape

    z = torch.arange(
        depth,
        device=flow_full.device,
        dtype=torch.float32,
    ).view(1, 1, depth, 1, 1)

    y = torch.arange(
        height,
        device=flow_full.device,
        dtype=torch.float32,
    ).view(1, 1, 1, height, 1)

    x = torch.arange(
        width,
        device=flow_full.device,
        dtype=torch.float32,
    ).view(1, 1, 1, 1, width)

    sampled_z = z + flow_full[:, 0:1].float()
    sampled_y = y + flow_full[:, 1:2].float()
    sampled_x = x + flow_full[:, 2:3].float()

    margin = float(margin)

    valid = (
        (sampled_z >= margin)
        & (sampled_z <= float(depth - 1) - margin)
        & (sampled_y >= margin)
        & (sampled_y <= float(height - 1) - margin)
        & (sampled_x >= margin)
        & (sampled_x <= float(width - 1) - margin)
    )

    return valid[0, 0].cpu().numpy().astype(bool)


def warp_mask_nearest(mask_np, flow_full):
    mask_tensor = torch.from_numpy(
        mask_np.astype(np.float32)
    ).unsqueeze(0).unsqueeze(0).to(DEVICE)

    depth, height, width = FULL_SIZE

    z = torch.arange(
        depth,
        device=DEVICE,
        dtype=torch.float32,
    )

    y = torch.arange(
        height,
        device=DEVICE,
        dtype=torch.float32,
    )

    x = torch.arange(
        width,
        device=DEVICE,
        dtype=torch.float32,
    )

    zz, yy, xx = torch.meshgrid(
        z, y, x,
        indexing="ij",
    )

    grid = torch.stack(
        [zz, yy, xx],
        dim=0,
    ).unsqueeze(0)

    grid = grid + flow_full.float()

    grid[:, 0] = (
        2.0 * grid[:, 0] / max(depth - 1, 1) - 1.0
    )
    grid[:, 1] = (
        2.0 * grid[:, 1] / max(height - 1, 1) - 1.0
    )
    grid[:, 2] = (
        2.0 * grid[:, 2] / max(width - 1, 1) - 1.0
    )

    grid = grid.permute(
        0, 2, 3, 4, 1
    )[..., [2, 1, 0]]

    warped = F.grid_sample(
        mask_tensor,
        grid,
        mode="nearest",
        padding_mode="zeros",
        align_corners=True,
    )

    return warped[0, 0].cpu().numpy() > 0.5


def folding_percent(jacobian, mask):
    if mask.sum() == 0:
        return np.nan

    return float(
        np.mean(jacobian[mask] <= 0.0) * 100.0
    )


# ------------------------------------------------------------
# 6. IC MSE：学習時と同じ定義
# ------------------------------------------------------------

z_low = torch.arange(
    LOW_SIZE[0],
    device=DEVICE,
    dtype=torch.float32,
)

y_low = torch.arange(
    LOW_SIZE[1],
    device=DEVICE,
    dtype=torch.float32,
)

x_low = torch.arange(
    LOW_SIZE[2],
    device=DEVICE,
    dtype=torch.float32,
)

zz_low, yy_low, xx_low = torch.meshgrid(
    z_low,
    y_low,
    x_low,
    indexing="ij",
)

LOW_BASE_GRID = torch.stack(
    [zz_low, yy_low, xx_low],
    dim=0,
).unsqueeze(0)


def valid_mask_low(flow_low):
    position = LOW_BASE_GRID + flow_low.float()

    return (
        (position[:, 0] >= 0)
        & (position[:, 0] <= LOW_SIZE[0] - 1)
        & (position[:, 1] >= 0)
        & (position[:, 1] <= LOW_SIZE[1] - 1)
        & (position[:, 2] >= 0)
        & (position[:, 2] <= LOW_SIZE[2] - 1)
    )


def masked_vector_mse(vector_field, valid_mask):
    per_voxel_mse = (
        vector_field.float()
        .square()
        .mean(dim=1)
    )

    valid_values = per_voxel_mse[valid_mask]

    if valid_values.numel() == 0:
        return torch.tensor(
            float("nan"),
            device=DEVICE,
        )

    return valid_values.mean()


def inverse_consistency_mse(
    flow_m2f_low,
    flow_f2m_low,
):
    # φ_MF + φ_FM(x + φ_MF)
    flow_f2m_on_m2f = transformer_low(
        flow_f2m_low.float(),
        flow_m2f_low.float(),
    )

    residual_m2f = (
        flow_m2f_low.float()
        + flow_f2m_on_m2f
    )

    mse_m2f = masked_vector_mse(
        residual_m2f,
        valid_mask_low(flow_m2f_low),
    )

    # φ_FM + φ_MF(x + φ_FM)
    flow_m2f_on_f2m = transformer_low(
        flow_m2f_low.float(),
        flow_f2m_low.float(),
    )

    residual_f2m = (
        flow_f2m_low.float()
        + flow_m2f_on_f2m
    )

    mse_f2m = masked_vector_mse(
        residual_f2m,
        valid_mask_low(flow_f2m_low),
    )

    return 0.5 * (mse_m2f + mse_f2m)


# ------------------------------------------------------------
# 7. 画像保存
# ------------------------------------------------------------

def save_pair_image(
    pair_name,
    moving_np,
    fixed_np,
    moved_np,
    jacobian,
    result,
):
    z = min(
        max(Z_CENTER, 0),
        FULL_SIZE[0] - 1,
    )

    vmin, vmax = np.percentile(
        np.concatenate(
            [
                moving_np[z].ravel(),
                fixed_np[z].ravel(),
                moved_np[z].ravel(),
            ]
        ),
        [1, 99],
    )

    difference = np.abs(fixed_np[z] - moved_np[z])

    difference_max = max(
        np.percentile(difference, 99),
        1e-6,
    )

    fig, axes = plt.subplots(
        1,
        5,
        figsize=(24, 5),
        constrained_layout=True,
    )

    panels = [
        (
            moving_np[z],
            "Moving",
            "gray",
            vmin,
            vmax,
        ),
        (
            fixed_np[z],
            "Fixed",
            "gray",
            vmin,
            vmax,
        ),
        (
            moved_np[z],
            "Moved\nCoarse-only LLL IC",
            "gray",
            vmin,
            vmax,
        ),
        (
            difference,
            "|Fixed - Moved|",
            "hot",
            0.0,
            difference_max,
        ),
        (
            jacobian[z],
            "Jacobian determinant\nred contour: ≤ 0",
            "coolwarm",
            -1.0,
            3.0,
        ),
    ]

    for axis, (
        image,
        title,
        cmap,
        panel_vmin,
        panel_vmax,
    ) in zip(axes, panels):

        axis.imshow(
            image,
            cmap=cmap,
            vmin=panel_vmin,
            vmax=panel_vmax,
        )

        if title.startswith("Jacobian"):
            axis.contour(
                jacobian[z] <= 0.0,
                levels=[0.5],
                colors="red",
                linewidths=0.7,
            )

        axis.set_title(title)
        axis.axis("off")

    fig.suptitle(
        f"{pair_name} | z={z} | "
        f"Dice={result['dice_after']:.4f} | "
        f"NCC={result['ncc_after']:.4f} | "
        f"interior Fold="
        f"{result['interior_valid_body_folding_percent']:.4f}% | "
        f"IC MSE={result['ic_mse']:.6f}",
        fontsize=13,
    )

    output_path = IMAGE_DIR / f"{pair_name}_z{z:03d}.png"

    fig.savefig(
        output_path,
        dpi=160,
        bbox_inches="tight",
    )

    plt.close(fig)

    return str(output_path)


# ------------------------------------------------------------
# 8. 1ペア評価
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_pair(pair_dir):
    fixed_path, moving_path = find_fixed_moving_paths(pair_dir)

    fixed_np = load_volume(fixed_path)
    moving_np = load_volume(moving_path)

    fixed_tensor = torch.from_numpy(
        fixed_np
    ).unsqueeze(0).unsqueeze(0).to(
        DEVICE,
        dtype=torch.float32,
    )

    moving_tensor = torch.from_numpy(
        moving_np
    ).unsqueeze(0).unsqueeze(0).to(
        DEVICE,
        dtype=torch.float32,
    )

    moving_bands = analysis(moving_tensor)
    fixed_bands = analysis(fixed_tensor)

    # Moving -> Fixed
    flow_m2f_low = model(
        moving_bands,
        fixed_bands,
    )

    flow_m2f_full = low_dvf_to_full_trilinear(
        flow_m2f_low
    )

    moved_tensor = transformer_full(
        moving_tensor,
        flow_m2f_full,
    )

    # Fixed -> Moving：IC MSE計算専用
    flow_f2m_low = model(
        fixed_bands,
        moving_bands,
    )

    ic_mse = float(
        inverse_consistency_mse(
            flow_m2f_low,
            flow_f2m_low,
        ).item()
    )

    moved_np = moved_tensor[0, 0].cpu().numpy()
    flow_np = flow_m2f_full[0].cpu().numpy()

    fixed_body = make_body_mask(fixed_np)
    moving_body = make_body_mask(moving_np)
    moved_body = make_body_mask(moved_np)

    fixed_lung = make_lung_mask(fixed_np)
    moving_lung = make_lung_mask(moving_np)

    moved_lung = warp_mask_nearest(
        moving_lung,
        flow_m2f_full,
    )

    jacobian = jacobian_determinant(flow_np)

    valid_body = (
        fixed_body
        & valid_reference_mask(
            flow_m2f_full,
            margin=0,
        )
    )

    interior_body = binary_erosion(
        fixed_body,
        iterations=INTERIOR_MARGIN_VOXELS,
    ).astype(bool)

    interior_valid_body = (
        interior_body
        & valid_reference_mask(
            flow_m2f_full,
            margin=INTERIOR_MARGIN_VOXELS,
        )
    )

    z_window = np.zeros_like(
        fixed_body,
        dtype=bool,
    )

    z0 = max(0, Z_START)
    z1 = min(FULL_SIZE[0] - 1, Z_END)

    z_window[z0:z1 + 1] = True

    interior_valid_body_z60_to_z68 = (
        interior_valid_body
        & z_window
    )

    mse_before = np.mean(
        (moving_np - fixed_np) ** 2
    )

    mse_after = np.mean(
        (moved_np - fixed_np) ** 2
    )

    ms_ssim_value, fsim_value = compute_ms_ssim_fsim(
        fixed_np,
        moved_np,
    )

    dice_before, jaccard_before = dice_jaccard(
        fixed_body,
        moving_body,
    )

    dice_after, jaccard_after = dice_jaccard(
        fixed_body,
        moved_body,
    )

    result = {
        "model": MODEL_NAME,
        "pair": pair_dir.name,
        "fixed_file": fixed_path.name,
        "moving_file": moving_path.name,

        "dice_before": dice_before,
        "dice_after": dice_after,

        "jaccard_before": jaccard_before,
        "jaccard_after": jaccard_after,

        "ncc_before": compute_ncc(
            fixed_np,
            moving_np,
        ),
        "ncc_after": compute_ncc(
            fixed_np,
            moved_np,
        ),

        "rmse_hu_before": float(
            np.sqrt(mse_before) * (HU_MAX - HU_MIN)
        ),
        "rmse_hu_after": float(
            np.sqrt(mse_after) * (HU_MAX - HU_MIN)
        ),

        "ssim_after": compute_mean_ssim(
            fixed_np,
            moved_np,
        ),
        "ms_ssim_after": ms_ssim_value,
        "fsim_after": fsim_value,

        "raw_full_folding_percent": float(
            np.mean(jacobian <= 0.0) * 100.0
        ),

        "valid_body_folding_percent": folding_percent(
            jacobian,
            valid_body,
        ),

        "interior_valid_body_folding_percent": folding_percent(
            jacobian,
            interior_valid_body,
        ),

        "interior_valid_body_folding_z60_to_z68_percent": (
            folding_percent(
                jacobian,
                interior_valid_body_z60_to_z68,
            )
        ),

        "jacobian_min": float(jacobian.min()),

        "lung_dice_after": dice_jaccard(
            fixed_lung,
            moved_lung,
        )[0],

        "lung_folding_percent": folding_percent(
            jacobian,
            fixed_lung,
        ),

        "valid_body_fraction_percent": float(
            100.0
            * valid_body.sum()
            / max(fixed_body.sum(), 1)
        ),

        "interior_valid_body_fraction_percent": float(
            100.0
            * interior_valid_body.sum()
            / max(fixed_body.sum(), 1)
        ),

        "flow_magnitude_mean_voxel": float(
            np.sqrt(
                np.sum(flow_np ** 2, axis=0)
            ).mean()
        ),

        # 学習時の inverse_consistency_loss_coarse と同じ定義
        "ic_mse": ic_mse,
    }

    if SAVE_ALL_PAIR_IMAGES:
        result["image_path"] = save_pair_image(
            pair_name=pair_dir.name,
            moving_np=moving_np,
            fixed_np=fixed_np,
            moved_np=moved_np,
            jacobian=jacobian,
            result=result,
        )

    del fixed_tensor
    del moving_tensor
    del moving_bands
    del fixed_bands
    del flow_m2f_low
    del flow_m2f_full
    del flow_f2m_low
    del moved_tensor

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ------------------------------------------------------------
# 9. 全101ペア評価
# ------------------------------------------------------------

pair_dirs = sorted(
    [
        path
        for path in TEST_DATA_ROOT.iterdir()
        if path.is_dir()
        and path.name.lower().startswith("pair")
    ],
    key=natural_sort_key,
)

if not pair_dirs:
    raise FileNotFoundError(
        f"pairフォルダがありません:\n{TEST_DATA_ROOT}"
    )

print(f"\n評価pair数: {len(pair_dirs)}")

rows = []

for index, pair_dir in enumerate(
    tqdm(
        pair_dirs,
        desc="Coarse-only LLL IC fine-tuned evaluation",
    ),
    start=1,
):
    try:
        result = evaluate_pair(pair_dir)
        rows.append(result)

        print(
            f"[{index:03d}/{len(pair_dirs):03d}] "
            f"{pair_dir.name} | "
            f"Dice={result['dice_after']:.5f} | "
            f"NCC={result['ncc_after']:.5f} | "
            f"rawFold={result['raw_full_folding_percent']:.4f}% | "
            f"interiorFold="
            f"{result['interior_valid_body_folding_percent']:.4f}% | "
            f"IC={result['ic_mse']:.6f}"
        )

    except Exception as error:
        print(
            f"[Skip] {pair_dir.name}: {repr(error)}"
        )


# ------------------------------------------------------------
# 10. Pair-wise CSV
# ------------------------------------------------------------

if len(rows) == 0:
    raise RuntimeError("評価成功pairが0件です。")

pairwise_df = pd.DataFrame(rows).sort_values(
    "pair",
    key=lambda column: column.map(natural_sort_key),
).reset_index(drop=True)

pairwise_csv = (
    EVAL_DIR
    / "pairwise_metrics_coarse_only_lll_ic_finetuned.csv"
)

pairwise_df.to_csv(
    pairwise_csv,
    index=False,
    encoding="utf-8-sig",
)

display(pairwise_df.round(6))

print("\nPair-wise CSV:")
print(pairwise_csv)


# ------------------------------------------------------------
# 11. Summary CSV
# ------------------------------------------------------------

raw_worst = pairwise_df.loc[
    pairwise_df[
        "raw_full_folding_percent"
    ].idxmax()
]

valid_worst = pairwise_df.loc[
    pairwise_df[
        "valid_body_folding_percent"
    ].idxmax()
]

interior_worst = pairwise_df.loc[
    pairwise_df[
        "interior_valid_body_folding_percent"
    ].idxmax()
]

z64_worst = pairwise_df.loc[
    pairwise_df[
        "interior_valid_body_folding_z60_to_z68_percent"
    ].idxmax()
]

summary_df = pd.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "n_pairs": int(len(pairwise_df)),

            "dice_before_mean": pairwise_df["dice_before"].mean(),
            "dice_after_mean": pairwise_df["dice_after"].mean(),
            "jaccard_after_mean": pairwise_df[
                "jaccard_after"
            ].mean(),

            "ncc_before_mean": pairwise_df["ncc_before"].mean(),
            "ncc_after_mean": pairwise_df["ncc_after"].mean(),

            "rmse_hu_before_mean": pairwise_df[
                "rmse_hu_before"
            ].mean(),
            "rmse_hu_after_mean": pairwise_df[
                "rmse_hu_after"
            ].mean(),

            "ssim_after_mean": pairwise_df["ssim_after"].mean(),
            "ms_ssim_after_mean": pairwise_df[
                "ms_ssim_after"
            ].mean(),
            "fsim_after_mean": pairwise_df["fsim_after"].mean(),

            "raw_full_folding_mean_percent": pairwise_df[
                "raw_full_folding_percent"
            ].mean(),
            "raw_full_folding_max_percent": pairwise_df[
                "raw_full_folding_percent"
            ].max(),
            "raw_full_worst_pair": raw_worst["pair"],

            "valid_body_folding_mean_percent": pairwise_df[
                "valid_body_folding_percent"
            ].mean(),
            "valid_body_folding_max_percent": pairwise_df[
                "valid_body_folding_percent"
            ].max(),
            "valid_body_worst_pair": valid_worst["pair"],

            "interior_valid_body_folding_mean_percent": pairwise_df[
                "interior_valid_body_folding_percent"
            ].mean(),
            "interior_valid_body_folding_max_percent": pairwise_df[
                "interior_valid_body_folding_percent"
            ].max(),
            "interior_valid_body_worst_pair": interior_worst["pair"],

            "interior_valid_body_folding_z60_to_z68_mean_percent": (
                pairwise_df[
                    "interior_valid_body_folding_z60_to_z68_percent"
                ].mean()
            ),
            "interior_valid_body_folding_z60_to_z68_max_percent": (
                pairwise_df[
                    "interior_valid_body_folding_z60_to_z68_percent"
                ].max()
            ),
            "interior_valid_body_folding_z60_to_z68_worst_pair": (
                z64_worst["pair"]
            ),

            "jacobian_min_mean": pairwise_df[
                "jacobian_min"
            ].mean(),
            "jacobian_global_min": pairwise_df[
                "jacobian_min"
            ].min(),

            "lung_dice_after_mean": pairwise_df[
                "lung_dice_after"
            ].mean(),
            "lung_folding_mean_percent": pairwise_df[
                "lung_folding_percent"
            ].mean(),

            "flow_magnitude_mean_voxel": pairwise_df[
                "flow_magnitude_mean_voxel"
            ].mean(),

            "ic_mse_mean": pairwise_df["ic_mse"].mean(),
            "ic_mse_std": pairwise_df["ic_mse"].std(),
            "ic_mse_max": pairwise_df["ic_mse"].max(),
            "ic_mse_worst_pair": pairwise_df.loc[
                pairwise_df["ic_mse"].idxmax(),
                "pair",
            ],
        }
    ]
)

summary_csv = (
    EVAL_DIR
    / "summary_metrics_coarse_only_lll_ic_finetuned.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False,
    encoding="utf-8-sig",
)

print("\n================ FINAL SUMMARY ================\n")
display(summary_df.round(6))
display(summary_df.T)

print("\nPair-wise CSV:")
print(pairwise_csv)

print("\nSummary CSV:")
print(summary_csv)

if SAVE_ALL_PAIR_IMAGES:
    print("\nSaved images:")
    print(IMAGE_DIR)

print("\nCompleted.")

Device     : cuda:0
TestData   : D:\Saito\Data\TestData
Model path : D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\model_coarse_only_lll_ic_finetuned_30k_final.pth
Output dir : D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\testdata_evaluation_coarse_only_lll_ic_finetuned_30k
Direction  : Moving -> Fixed
[64, 128, 128]

Model loading completed.
Checkpoint epoch: 30000
Architecture    : unknown
Loss            : unknown

評価pair数: 101


Coarse-only LLL IC fine-tuned evaluation:   0%|          | 0/101 [00:00<?, ?it/s]

[001/101] pair1 | Dice=0.99386 | NCC=0.99263 | rawFold=1.8430% | interiorFold=1.8255% | IC=0.252458
[002/101] pair2 | Dice=0.99327 | NCC=0.99259 | rawFold=1.0931% | interiorFold=1.7712% | IC=0.231494
[003/101] pair3 | Dice=0.99382 | NCC=0.99203 | rawFold=1.3930% | interiorFold=2.2932% | IC=0.214796
[004/101] pair4 | Dice=0.99378 | NCC=0.99185 | rawFold=1.0766% | interiorFold=2.5216% | IC=0.196643
[005/101] pair5 | Dice=0.99408 | NCC=0.99115 | rawFold=1.1817% | interiorFold=2.7925% | IC=0.204965
[006/101] pair6 | Dice=0.99464 | NCC=0.99356 | rawFold=0.7456% | interiorFold=1.5710% | IC=0.222554
[007/101] pair7 | Dice=0.99466 | NCC=0.99232 | rawFold=1.7324% | interiorFold=2.0568% | IC=0.233474
[008/101] pair8 | Dice=0.99455 | NCC=0.99155 | rawFold=2.2483% | interiorFold=2.5043% | IC=0.263572
[009/101] pair9 | Dice=0.99350 | NCC=0.99256 | rawFold=0.7309% | interiorFold=1.5270% | IC=0.156748
[010/101] pair10 | Dice=0.99302 | NCC=0.99300 | rawFold=0.7920% | interiorFold=1.5323% | IC=0.160426

,model,pair,fixed_file,moving_file,dice_before,dice_after,jaccard_before,jaccard_after,ncc_before,ncc_after,...,interior_valid_body_folding_percent,interior_valid_body_folding_z60_to_z68_percent,jacobian_min,lung_dice_after,lung_folding_percent,valid_body_fraction_percent,interior_valid_body_fraction_percent,flow_magnitude_mean_voxel,ic_mse,image_path
0,30k_coarse_only_lll_ic_finetuned,pair1,fixed_masked_B_MIC00197_2019_0406.npz,registered_masked_A_MIC00197_2018_0428.npz,0.961797,0.993859,0.926405,0.987793,0.928469,0.992631,...,1.825451,1.007682,-8.032995,0.904579,1.573820,99.944213,83.291195,1.391269,0.252458,D:\Saito\fullres_trilinear_ic_beta1e3_80k\coar...
1,30k_coarse_only_lll_ic_finetuned,pair2,fixed_masked_B_MIC00761_2020_0926.npz,registered_masked_A_MIC00761_2019_0727.npz,0.981110,0.993269,0.962920,0.986628,0.962952,0.992588,...,1.771180,1.573112,-9.117491,0.909043,1.884039,99.891662,81.989555,1.304247,0.231494,D:\Saito\fullres_trilinear_ic_beta1e3_80k\coar...
2,30k_coarse_only_lll_ic_finetuned,pair3,fixed_masked_B_MIC03568_2019_0824.npz,registered_masked_A_MIC03568_2018_0825.npz,0.963118,0.993822,0.928860,0.987720,0.928717,0.992027,...,2.293226,2.473479,-7.080524,0.843842,2.316654,99.946054,82.930456,1.442814,0.214796,D:\Saito\fullres_trilinear_ic_beta1e3_80k\coar...
3,30k_coarse_only_lll_ic_finetuned,pair4,fixed_masked_B_MIC03568_2020_0829.npz,registered_masked_A_MIC03568_2018_0825.npz,0.982659,0.993777,0.965908,0.987631,0.957978,0.991850,...,2.521586,1.454833,-4.999599,0.855031,3.972035,99.916393,81.827218,1.338172,0.196643,D:\Saito\fullres_trilinear_ic_beta1e3_80k\coar...
4,30k_coarse_only_lll_ic_finetuned,pair5,fixed_masked_B_MIC03568_2020_0829.npz,registered_masked_A_MIC03568_2019_0824.npz,0.986525,0.994083,0.973408,0.988236,0.959973,0.991150,...,2.792461,2.104667,-4.795894,0.841360,4.647542,99.918090,81.614067,1.441799,0.204965,D:\Saito\fullres_trilinear_ic_beta1e3_80k\coar...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,30k_coarse_only_lll_ic_finetuned,pair97,fixed_masked_B_MIC45438_2020_0824.npz,registered_masked_A_MIC45438_2019_0903.npz,0.979977,0.994330,0.960741,0.988723,0.947270,0.993219,...,1.462433,1.475213,-2.720105,0.771784,1.988350,99.911700,82.546278,1.884112,0.203181,D:\Saito\fullres_trilinear_ic_beta1e3_80k\coar...
97,30k_coarse_only_lll_ic_finetuned,pair98,fixed_masked_B_MIC52743_2020_0125.npz,registered_masked_A_MIC52743_2018_1213.npz,0.978022,0.994569,0.956990,0.989197,0.952752,0.993387,...,1.924198,2.139693,-5.164343,0.876141,2.228216,99.913081,82.605379,1.640482,0.185976,D:\Saito\fullres_trilinear_ic_beta1e3_80k\coar...
98,30k_coarse_only_lll_ic_finetuned,pair99,fixed_masked_B_MIC53652_2020_1219.npz,registered_masked_A_MIC53652_2019_1213.npz,0.981859,0.994742,0.964364,0.989540,0.947066,0.992336,...,1.598048,0.983446,-5.274282,0.849029,1.980716,99.955748,83.444353,1.711398,0.251470,D:\Saito\fullres_trilinear_ic_beta1e3_80k\coar...
99,30k_coarse_only_lll_ic_finetuned,pair100,fixed_masked_B_MIC53770_2019_1111.npz,registered_masked_A_MIC53770_2018_1128.npz,0.980435,0.993628,0.961621,0.987336,0.954214,0.992794,...,1.167034,0.524086,-4.290873,0.939365,0.875135,99.888047,83.098710,1.271413,0.163371,D:\Saito\fullres_trilinear_ic_beta1e3_80k\coar...



Pair-wise CSV:
D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\testdata_evaluation_coarse_only_lll_ic_finetuned_30k\pairwise_metrics_coarse_only_lll_ic_finetuned.csv

================ FINAL SUMMARY ================



,model,n_pairs,dice_before_mean,dice_after_mean,jaccard_after_mean,ncc_before_mean,ncc_after_mean,rmse_hu_before_mean,rmse_hu_after_mean,ssim_after_mean,...,interior_valid_body_folding_z60_to_z68_worst_pair,jacobian_min_mean,jacobian_global_min,lung_dice_after_mean,lung_folding_mean_percent,flow_magnitude_mean_voxel,ic_mse_mean,ic_mse_std,ic_mse_max,ic_mse_worst_pair
0,30k_coarse_only_lll_ic_finetuned,101,0.959418,0.988571,0.978007,0.920009,0.983142,411.771446,171.831054,0.892246,...,pair84,-7.851125,-21.13275,0.79859,2.741671,1.764373,0.519343,0.995228,6.26288,pair78


,0
model,30k_coarse_only_lll_ic_finetuned
n_pairs,101
dice_before_mean,0.959418
dice_after_mean,0.988571
jaccard_after_mean,0.978007
ncc_before_mean,0.920009
ncc_after_mean,0.983142
rmse_hu_before_mean,411.771446
rmse_hu_after_mean,171.831054
ssim_after_mean,0.892246



Pair-wise CSV:
D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\testdata_evaluation_coarse_only_lll_ic_finetuned_30k\pairwise_metrics_coarse_only_lll_ic_finetuned.csv

Summary CSV:
D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\testdata_evaluation_coarse_only_lll_ic_finetuned_30k\summary_metrics_coarse_only_lll_ic_finetuned.csv

Saved images:
D:\Saito\fullres_trilinear_ic_beta1e3_80k\coarse_only_lll_ic_finetune_30k\testdata_evaluation_coarse_only_lll_ic_finetuned_30k\pair_images

Completed.
